# Learning Reinforcement Learning
## Einführung in das Notizbuch
### In diesem Notizbuch werden die grundlegenden Reinforcement Learning Prinzipien erklärt und veranschaulicht, die ich betrachten und verstehen muss, um die Projektidee in einem für mich neuen Themengebiet umzusetzen.
---

## Inhaltsverzeichnis
1. [Markov Decision Process (MDP)](#1-markov-decision-process-mdp)
2. [Policy π](#2-policy-π)
3. [Value Function & Q-Function](#3-value-function--q-function)
4. [Bellman Equation](#4-bellman-equation)
5. [DQN - Deep Q-Network](#5-dqn--dep-q-network)
6. [A2C - Advantage Actor-Critic](#6-a2c--advantage-actor-critic)
7. [PPO - Procimal Policy Optimization](#7-ppo--proximal-policy-optimization)
8. [Exploration vs. Exploitation](#8-exploration-vs-exploitation)
9. [Weitere Ressourcen & Lernmaterial](#9-weitere-ressourcen--lernmaterialien)


---

## 1. Markov Decision Process (MDP)
### Was ist ein MDP?
Ein MDP ist an sich das mathematische Regelwerk und beschreibt wie ein Agent mit seiner Umgebung interagiert.
Alles was ich in meinem Projekt mache (die Agenten die durchs Maze navigieren) lassen sich auf dieses Modell zurückführen.

Formal ist ein MDP ein Tupel asus fünf Elementen:

> **MDP = (S, A, P, R, y)** 

### Erklärung der Symbole:
- S (State Space) = Alle möglichen Zustände >> Position des Agenten im Maze
- A (Action Space) = Alle möglichen Aktionen >> oben, oben-rechts, rechts, unten-rechts, unten, unten-links, links, oben-links
- P (Transition Function) = Übergangswahrscheinlichkeit >> Determinitstisch in der Simulation
- R (Reward Function) = R(s,a,s') - Belohnung pro Tranisition
- y (Diskontfaktor) = Gewichtung zukünftiger Rewards z.b. 0.99 >> Agent denkt langfristig

### Die Markov-Eigenschaft:
Der Agent trifft an sich Entscheidungen ausschließlich basierend auf dem aktuellen State s -> Vergangenheit spielt keine Rolle

### Der MDP-Loop - State - Action - Reward:
S0 -> A0 -> R1 -> S1 -> A1 -> R2 -> S2 -> A2 -> R3 -> ... 

### Unterscheidung MDP und POMDP
In meiner Unity-Simulation hat der Agent eine vollständige Sichtbarkeit über seine Position und Umgebung >> Erfüllung der MDP-Bedingung
In der Realität würde man für ein Lagerroboter mit Sensorrauschen POMDP (Partially Observable MDP) nutzen. Für mein Projekt ist das hier eine bewusste Vereinfachung und eine Limitation die dikutiert werden wird.

### Quellen:
- Part 1: Key Concepts in RL: https://spinningup.openai.com/en/latest/spinningup/rl_intro.html
- Leccture 2: Markov Decision Processes: https://web.stanford.edu/class/cme241/lecture_slides/david_silver_slides/MDP.pdf

In [5]:
import gymnasium as gym
import numpy as np

# FrozenLake als MDP Beispiel, 
# mit is_slippery = false (weil deterministisch)
env = gym.make("FrozenLake-v1", is_slippery=False)
obs, info = env.reset(seed = 42)

print("S - STATE SPACE")
print(f"State Space (S): {env.observation_space}")
print(f"Startzustand s0 = {obs}")

print("\n A - ACTION SPACE")
print(f"Action Space (A): {env.action_space}")
action_names = {0: "links", 1: "unten", 2: "rechts", 3: "oben"}
for idx, name in action_names.items():
    print(f"   Aktion {idx}: {name}")

# P(s'|s,a) - Übergangsfunktion für State 0
# komplette Tranitionsmatric wird durch env.unwrapped.P
print(f"\n P(s'|s,a) - Übergangsfunktion für State 0")
print("(probability, next_state, reward, terminated)")
for action_idx, action_name in action_names.items():
    transitions = env.unwrapped.P[0][action_idx]          # alle Transitionen von State 0
    print(f" Aktion '{action_name}': {transitions}")
print(">> Bei is_slippery=False: Wahrscheinlichkeit immer 1.0 = deterministisch")
print(">> Bei is_slippery=True:  Wahrscheinlichkeiten verteilt = stochastisch")

# R(s,a,s') - mögliche Reward-Werte in FrozenLake
print(f"\n R(s,a,s') - Reward-Funktion vollständig zeigen")
print("Alle möglichen Rewards im FrozenLake:")
rewards_seen = {}
for state in range(env.observation_space.n):               # alle States mal durchgehen
    for action in range(env.action_space.n):               # mal alle Aktionen durchgehen
        for prob, next_state, reward, terminated in env.unwrapped.P[state][action]:
            rewards_seen[reward] = rewards_seen.get(reward, 0) + 1

for reward_val, count in sorted(rewards_seen.items()):
    print(f"  Reward {reward_val}: kommt {count}x vor in der Transitionsmatrix")


print("\nIn meinem Unity-Projekt (formal definiert (wahrscheinlich) in Kapitel 2):")
print("  R(s,a,s') = +10.0  wenn s' = Zielzustand")
print("  R(s,a,s') =  -1.0  wenn s' = Wand (Agent prallt ab)")
print("  R(s,a,s') =  -0.01 pro Schritt (Step Penalty)")
print("  R(s,a,s') =   0.0  sonst")

# konkrete Transition als Beispiel
print("\n Konkrete Transition - Beispiel")
obs, info = env.reset(seed=42)
action = 2                                                 # rechts gehen
next_obs, reward, terminated, truncated, info = env.step(action)
print(f"s  = {obs} (Startzustand)")
print(f"a  = {action} ({action_names[action]})")
print(f"s' = {next_obs} (neuer Zustand)")
print(f"R(s,a,s') = {reward}")
print(f"Terminated: {terminated}")

# Diskontfaktor γ
print("\n Diskontfaktor γ = 0.99")
gamma = 0.99
print(f"{'Schritt t':<12} {'γᵗ':<12} {'Reward noch X% wert'}")
print("-" * 40)
for t in [0, 1, 5, 10, 50, 100]:
    print(f"{t:<12} {gamma**t:<12.4f} {gamma**t*100:.2f}%")
print(f"\n>> γ=0.99: Agent denkt sehr langfristig — sinnvoll wenn Ziel weit entfernt")

# Hinweis: Unterschied FrozenLake vs. Unity-Maze 
print("\n Hinweis: FrozenLake vs. mein Unity-Maze")
print("Konzeptionell ähnlich:")
print("  ✓ Beide haben klares Ziel")
print("  ✓ Beide haben Bestrafung für schlechte Aktionen")
print("  ✓ Beide nutzen diskrete Aktionen")
print("  ✓ Markov-Eigenschaft gilt in beiden")
print("\nStrukturelle Unterschiede:")
print("  ✗ FrozenLake: diskrete Zustände (0-15)")
print("  ✗ Unity-Maze: kontinuierliche Positionen, diskrete Aktionen (8 Richtungen)")
print("  ✗ FrozenLake: 4 Aktionen — Unity-Maze: 8 Aktionen")

env.close()

S - STATE SPACE
State Space (S): Discrete(16)
Startzustand s0 = 0

 A - ACTION SPACE
Action Space (A): Discrete(4)
   Aktion 0: links
   Aktion 1: unten
   Aktion 2: rechts
   Aktion 3: oben

 P(s'|s,a) - Übergangsfunktion für State 0
(probability, next_state, reward, terminated)
 Aktion 'links': [(1.0, 0, 0.0, False)]
 Aktion 'unten': [(1.0, 4, 0.0, False)]
 Aktion 'rechts': [(1.0, 1, 0.0, False)]
 Aktion 'oben': [(1.0, 0, 0.0, False)]
>> Bei is_slippery=False: Wahrscheinlichkeit immer 1.0 = deterministisch
>> Bei is_slippery=True:  Wahrscheinlichkeiten verteilt = stochastisch

 R(s,a,s') - Reward-Funktion vollständig zeigen
Alle möglichen Rewards im FrozenLake:
  Reward 0.0: kommt 63x vor in der Transitionsmatrix
  Reward 1.0: kommt 1x vor in der Transitionsmatrix

In meinem Unity-Projekt (formal definiert (wahrscheinlich) in Kapitel 2):
  R(s,a,s') = +10.0  wenn s' = Zielzustand
  R(s,a,s') =  -1.0  wenn s' = Wand (Agent prallt ab)
  R(s,a,s') =  -0.01 pro Schritt (Step Penalty)
  R

## 9. Weitere Ressourcen & Lernmaterialien
### Die folgenden Quellen dienen als Orientierung beim Einstieg in Reinforcement Learning. Sie werden nicht als zitierfähige Quellen in der wissenschaftlichen Arbeit verwendet, sondern sind eher für den persönlichen Lernprozesse und um tiefer in die Materie einzusteigen:
- RL from Scratch - Learning Roadmap: https://jugalgajjar.github.io/RL-From-Scratch/roadmap
- RLStudyGuide - How to Learn RL Step by Step: https://github.com/anhOfTheStars/RLStudyGuide?tab=readme-ov-file
- AI/ML Roadmap - Module 9: Reinforcement Learning: https://deepwiki.com/aadi1011/AI-ML-Roadmap-from-scratch/2.10-module-9:-reinforcement-learning
- A visual guide on Reiforcement Learning - the 6 things that makes it "click" (Youtube-Video): https://www.youtube.com/watch?v=Qpx6WD0qekQ